# Dentate RL bootcamp on Colab

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Xpitfire/dentate-bootcamp/blob/main/dentate_bootcamp.ipynb)

This notebook installs `dentate[demo]`, materializes the bundled starter project and pinned SmolLM tokenizer,
trains a fresh ~0.1M-core model on CPU (SFT, measured support, gated GRPO, frozen evaluation), then starts the
Dentate site in the background and opens it through Colab's port proxy so you can inspect the result in the lab UI.
The demo ships Spiral, a small looped transformer, as its reference architecture; any architecture registered with
Dentate runs through the same pipeline.

Everything runs offline after the install: no model weights are downloaded and no teacher model is called. A failed
support gate is a valid result; cold small models often fail it, and post-GRPO accuracy is then absent rather than zero.

The notebook also runs headless (`jupyter nbconvert --execute`) on plain Linux: every Colab-specific call is guarded,
and the server starts only when a proxy is available.


In [ ]:
# 1. Install. Colab ships torch already; elsewhere the `demo` extra pulls in a CPU torch.
import importlib.util, subprocess, sys

def pip(*args):
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", *args], check=True)

if importlib.util.find_spec("dentate") is None:
    pip("dentate[demo]")
DENTATE = [sys.executable, "-m", "dentate.cli"]      # the `dentate` console script, pinned to THIS kernel's interpreter
print("dentate installed:", subprocess.run([*DENTATE, "--help"], capture_output=True, text=True).stdout.splitlines()[0])


In [ ]:
# 2. Materialize the demo bundle (tokenizer + starter project) under ~/.dentate/demo (or $DENTATE_HOME/demo) and pin DENTATE_TOKENIZER_DIR.
import os, subprocess
subprocess.run([*DENTATE, "demo", "doctor"], check=True)
init = subprocess.run([*DENTATE, "demo", "init"], check=True, capture_output=True, text=True).stdout
print(init)
for line in init.splitlines():
    if line.startswith("export "):
        key, _, value = line[len("export "):].partition("=")
        os.environ[key] = value           # the CLI sets it for ITS process; propagate the export line into this kernel
os.environ.setdefault("HF_HUB_OFFLINE", "1")
os.environ.setdefault("TRANSFORMERS_OFFLINE", "1")

In [ ]:
# 3. Train the starter project. Progress lines are JSON (one per stage); the last line names the result package.
#    ~2-6 min on a Colab CPU runtime. Edit `~/.dentate/demo/starter.dentate` (steps, width, loops) to experiment.
import json, os, pathlib, subprocess, time

HOME = pathlib.Path(os.environ.get("DENTATE_HOME") or pathlib.Path.home() / ".dentate")   # same root the CLI uses
OUT = HOME / "experiments" / time.strftime("colab-%Y%m%d-%H%M%S")
proc = subprocess.Popen([*DENTATE, "demo", "run", "--out", str(OUT)], stdout=subprocess.PIPE, text=True, bufsize=1)
stage = None
for line in proc.stdout:
    try:
        state = json.loads(line)
    except ValueError:
        print(line.rstrip()); continue
    if "message" in state:                                   # trainer log lines (loss / accuracy per iteration)
        print(f"[{state.get('elapsed_seconds', 0):>7.1f}s] {state['message']}")
    elif "out" in state:                                     # the final summary line
        print(f"finished: {state['status']} → {state['out']}")
    elif state.get("stage") != stage:                        # stage transitions: sft → support → grpo → frozen_evaluation
        stage = state.get("stage")
        print(f"[{state.get('elapsed_seconds', 0):>7.1f}s] stage: {stage}")
assert proc.wait() == 0, "demo run failed — see the log above"
results = json.loads((OUT / "results.json").read_text())
print("\nstatus:", results["status"])
before, after = results.get("frozen_before") or {}, results.get("frozen_after") or {}
print("gate verdict      :", (results.get("gate") or {}).get("verdict"))
print("frozen eval before GRPO:", before.get("verify_acc"))
print("frozen eval after  GRPO:", after.get("verify_acc", "absent (gate failed or GRPO did not run)"))
print("result package    :", OUT / "result.dentate")

In [ ]:
# 4. Start the Dentate site in the background (public pages + your local lab) and open it through Colab's port proxy.
#    Outside Colab this cell only starts the server when the port is free and prints the local URL.
import socket, subprocess, sys, time

PORT = 8793
def listening(port):
    with socket.socket() as s:
        return s.connect_ex(("127.0.0.1", port)) == 0

server = None
if not listening(PORT):
    server = subprocess.Popen([*DENTATE, "serve", "--port", str(PORT)], stdout=subprocess.DEVNULL, stderr=subprocess.STDOUT)
    for _ in range(60):
        if listening(PORT):
            break
        time.sleep(0.5)
assert listening(PORT), "dentate serve did not come up on port %d" % PORT

try:
    from google.colab.output import eval_js                      # Colab only
    url = eval_js(f"google.colab.kernel.proxyPort({PORT})")
    from IPython.display import HTML, IFrame, display
    display(HTML(f'<a href="{url}" target="_blank"><b>Open Dentate in a new tab: {url}</b></a>'))
    display(IFrame(url, width="100%", height=720))
except ImportError:
    print(f"Not on Colab — open http://127.0.0.1:{PORT} in your browser.")

## Next steps

* In the site's **Lab** tab, import `result.dentate` (or edit the starter project) and launch a new experiment; toggle it
  **public** to share a results page.
* Locally: `pip install "dentate[demo]" && dentate demo init && dentate demo run && dentate serve` runs the same package
  and the same site; your data stays under `~/.dentate`.
* Hosted: https://dentate.cortex.a2olabs.com (sign in to launch experiments on the bootcamp workers).
* Student guide: [BOOTCAMP.md](https://github.com/Xpitfire/dentate-bootcamp/blob/main/BOOTCAMP.md).


In [ ]:
# 5. (optional) Stop the background server when you are done.
if 'server' in globals() and server is not None and server.poll() is None:
    server.terminate()
    print("server stopped")